# Plugin Metadata

> Data structures for plugin metadata

In [ ]:
#| default_exp core.metadata

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

## PluginMeta

The `PluginMeta` dataclass stores metadata about a plugin, including its name, version, and runtime state.

In [ ]:
#| export
@dataclass
class PluginTaxonomy:
    """Derived classification of a plugin via its interface FQN.
    
    Populated at install time by `_generate_manifest`'s introspection script
    (running in the plugin's own conda env, where the interface library is
    installed). Substrate stores strings only — no host-side imports of
    interface libraries needed for taxonomy queries.
    
    - `domain`: the substrate area, derived from the interface library's
      naming convention `cjm-<domain>-plugin-system` (e.g., "transcription",
      "graph", "media", "text").
    - `role`: the interface class name segment of the FQN (e.g.,
      "TranscriptionPlugin", "GraphPlugin", "ForcedAlignmentPlugin"). Multiple
      plugins can share a role within a domain.
    - `interface_fqcn`: the full dotted path, kept verbatim for queries and
      reverse-lookup.
    """
    domain: str  # e.g., "transcription", "graph", "media"
    role: str  # e.g., "TranscriptionPlugin", "GraphPlugin"
    interface_fqcn: str  # Full dotted interface class path


@dataclass
class ResourceRequirements:
    """Binary hard-facts about what a plugin needs to run (Phase 5a).
    
    Quantitative resource amounts (min_vram_mb, etc.) deliberately omitted
    per CR-7's reactive resource management reframing — plugin authors can't
    reliably estimate model × dtype × quantization combinatorics, and Blender-
    style variable-render plugins can't estimate at all. The substrate uses
    these binary hard-facts purely for discovery filtering; actual resource
    contention is handled reactively by CR-7's eviction + retry flow.
    
    - `requires_gpu`: True if the plugin needs any GPU; the substrate gates
      execution on a system monitor reporting one is present.
    - `platforms`: e.g., ["linux-x64", "darwin-arm64"]. Empty list means no
      platform constraint declared (assume universal compatibility).
    - `accelerators`: e.g., ["cuda", "mps", "cpu"]. Informational; substrate
      doesn't auto-select but consumers can filter on the values.
    """
    requires_gpu: bool = False
    platforms: List[str] = field(default_factory=list)
    accelerators: List[str] = field(default_factory=list)

In [ ]:
#| export
@dataclass
class PluginMeta:
    """Metadata about a plugin."""
    name:str # Plugin's unique identifier
    version:str # Plugin's version string
    description:str="" # Brief description of the plugin's functionality
    # SG-35: `author` and `package_name` removed — author lives in pyproject.toml
    # + importlib.metadata; package_name is derivable from the import system.
    # `description` is retained and validated by SG-6's manifest checker.
    category:str="" # Display label — derived from taxonomy.domain (CR-1) or read from legacy manifests
    interface:str="" # Fully qualified interface class name (kept verbatim for SG-7 format check)
    # CR-1: taxonomy block supersedes the legacy flat `category` string with a
    # derived domain/role/interface_fqcn triple. Populated at discovery from
    # the manifest's taxonomy block (which `_generate_manifest` emits at install
    # time). Optional during the SG-47 cascade window: legacy manifests without
    # the block read taxonomy=None and rely on the legacy `category` string.
    taxonomy:Optional["PluginTaxonomy"]=None
    # Phase 5a: binary hard-facts for discovery filtering (no quantitative amounts
    # per CR-7 reactive reframing). Optional during the cascade window for the
    # same reason as taxonomy.
    resources:Optional["ResourceRequirements"]=None
    config_schema:Optional[Dict[str, Any]]=None # JSON Schema for plugin configuration
    instance:Optional[Any]=None # Plugin instance (PluginInterface subclass)
    enabled:bool=True # Whether the plugin is enabled
    last_executed:float=0.0 # Unix timestamp
    # SG-9: drift detection — set by PluginManager.load_plugin when the live
    # worker's /config_schema disagrees with the manifest's stored config_schema.
    # `live_config_schema` holds the worker-reported shape so callers can pick
    # which to honor (substrate keeps using `config_schema` for defaults +
    # validation; tooling and the future plugin-config UI library can inspect
    # `live_config_schema` for the post-regenerate-manifest preview).
    config_schema_drift:bool=False
    live_config_schema:Optional[Dict[str, Any]]=None

### Example: Creating Plugin Metadata

In [ ]:
# Create plugin metadata
meta = PluginMeta(
    name="example_plugin",
    version="1.0.0",
    description="An example plugin",
    category="transcription",
    interface="cjm_transcription_plugin_system.plugin_interface.TranscriptionPlugin",
    config_schema={
        "type": "object",
        "properties": {
            "model": {"type": "string", "default": "base"},
            "device": {"type": "string", "enum": ["cpu", "cuda"]}
        }
    }
)

print("PluginMeta instance:")
print(meta)
print(f"\nName: {meta.name}")
print(f"Version: {meta.version}")
print(f"Category: {meta.category}")
print(f"Interface: {meta.interface}")
print(f"Config Schema: {meta.config_schema}")
print(f"Enabled: {meta.enabled}")
print(f"Instance: {meta.instance}")

In [ ]:
# Test with minimal arguments
minimal_meta = PluginMeta(name="minimal", version="0.1.0")
print(f"Minimal PluginMeta: {minimal_meta}")

# Test equality
meta_copy = PluginMeta(name="minimal", version="0.1.0")
print(f"\nEquality test: {minimal_meta == meta_copy}")

Minimal PluginMeta: PluginMeta(name='minimal', version='0.1.0', description='', author='', package_name='', category='', interface='', config_schema=None, instance=None, enabled=True, last_executed=0.0)

Equality test: True


In [ ]:
# CR-1 + Phase 5a: PluginTaxonomy and ResourceRequirements round-trip cleanly,
# and PluginMeta accepts either or both as optional fields.
tax = PluginTaxonomy(
    domain="transcription",
    role="TranscriptionPlugin",
    interface_fqcn="cjm_transcription_plugin_system.plugin_interface.TranscriptionPlugin",
)
assert tax.domain == "transcription"
assert tax.role == "TranscriptionPlugin"

res = ResourceRequirements(
    requires_gpu=True,
    platforms=["linux-x64", "darwin-arm64"],
    accelerators=["cuda", "mps"],
)
assert res.requires_gpu is True
assert "linux-x64" in res.platforms

# ResourceRequirements defaults: no GPU, empty platforms (universal), empty accelerators.
default_res = ResourceRequirements()
assert default_res.requires_gpu is False
assert default_res.platforms == []
assert default_res.accelerators == []

# PluginMeta accepts taxonomy + resources; both default to None for legacy manifests.
typed_meta = PluginMeta(
    name="whisper-local",
    version="1.0.0",
    description="Local Whisper-based STT",
    interface="cjm_transcription_plugin_system.plugin_interface.TranscriptionPlugin",
    taxonomy=tax,
    resources=res,
)
assert typed_meta.taxonomy.role == "TranscriptionPlugin"
assert typed_meta.resources.requires_gpu is True

legacy_meta = PluginMeta(name="legacy", version="0.0.1")
assert legacy_meta.taxonomy is None
assert legacy_meta.resources is None

print("✓ PluginTaxonomy + ResourceRequirements + PluginMeta integration")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()